# 20 - Production Deployment Setup

**Nginx + Uvicorn + Docker Compose + Structured Logging**

---

## Cel

Pokazać **production-ready setup** dla FastAPI:
- **Nginx** jako reverse proxy
- **Uvicorn** z multiple workers
- **Docker Compose** orkiestracja
- **Structured Logging** (JSON)
- **Health checks** i monitoring basics

**Uwaga:** Kod aplikacji (main.py) jest prosty (CRUD), żeby skupić się na infrastrukturze deployment.

---

## Architektura

```
Internet → Nginx (port 8080) → FastAPI/Uvicorn (port 8000)
            │                      │
            └─ JSON logs ──────────┴─ JSON logs
```

**Przepływ requestu:**
1. Klient wysyła request do `http://localhost:8080/tasks`
2. Nginx odbiera na porcie 8080 (host) → 80 (kontener)
3. Nginx przekazuje do FastAPI (reverse proxy) `app:8000`
4. FastAPI przetwarza request (1 z 2 workers)
5. Response wraca przez nginx do klienta

**Komponenty:**
- `nginx/` - Reverse proxy config
- `main.py` - FastAPI app (CRUD + /health)
- `Dockerfile` - Multi-stage build
- `docker-compose.yml` - Orkiestracja
- `logs/` - JSON logs (volume)

---

## Networking - Reverse Proxy, Port Forwarding, Bridge

### 3 różne rzeczy które się dzieją:

#### 1. **Reverse Proxy** (nginx) ← Warstwa aplikacji (HTTP)

```
Client → Nginx (reverse proxy) → Backend (FastAPI)
         ^^^^^^^^^^^^^^^^^^^^^^
         Przekazuje requesty HTTP
```

**Co to robi:**
- Odbiera request HTTP od klienta
- Przekazuje go do backendu (`proxy_pass http://app:8000`)
- Zwraca response do klienta

**nginx.conf:**
```nginx
upstream fastapi_backend {
    server app:8000;  # Backend
}

location / {
    proxy_pass http://fastapi_backend;  # ← REVERSE PROXY
    proxy_set_header Host $host;
}
```

**Nazwa techniczna:** Reverse Proxy, HTTP Proxy, Proxy Pass

---

#### 2. **Port Forwarding** (Docker) ← Warstwa transportu (TCP)

```
Host port 8080 → Container port 80 (nginx)
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Mapowanie portów
```

**docker-compose.yml:**
```yaml
nginx:
  ports:
    - "8080:80"  # ← PORT FORWARDING
    #  ^^^^  ^^     host:container
    #  Host  Container
```

**Co to znaczy `8080:80`?**
- Nginx **wewnątrz** kontenera słucha na **porcie 80**
- Docker **mapuje** to na **port 8080** hosta
- `curl localhost:8080` → trafia do nginx:80 w kontenerze

**Dlaczego 8080, nie 80?**
- Port 80 może być zajęty systemowo (Apache/nginx na hoście)
- W produkcji (bez konfliktu): `80:80` lub `443:443` (HTTPS)

**Nazwa techniczna:** Port Forwarding, Port Mapping, Port Publishing

---

#### 3. **Bridge Network** (Docker) ← Warstwa sieciowa (IP)

```
nginx container → app_network (bridge) → app container
                  ^^^^^^^^^^^^^^^^^^^^
                  Wirtualna sieć Docker
```

**docker-compose.yml:**
```yaml
networks:
  app_network:
    driver: bridge  # ← BRIDGE NETWORK

services:
  nginx:
    networks:
      - app_network  # Nginx w sieci
  
  app:
    networks:
      - app_network  # App w tej samej sieci
```

**Co to robi:**
- Kontenery w tej samej sieci bridge mogą się komunikować
- Nginx może połączyć się z `app:8000` (DNS name)
- Host **NIE MOŻE** połączyć się z `app:8000` (nie jest published)

**Nazwa techniczna:** Bridge Network, Container Networking, Docker Network

---

### Podsumowanie różnic:

| Co? | Warstwa OSI | Nazwa | Co robi? |
|-----|-------------|-------|----------|
| Nginx przekazuje HTTP | L7 (Aplikacja) | **Reverse Proxy** | HTTP request forwarding |
| Docker mapuje porty | L4 (Transport) | **Port Forwarding** | `8080:80` (host→container) |
| Docker łączy kontenery | L3 (Sieć) | **Bridge Network** | nginx ↔ app komunikacja |

---

### Cały przepływ requestu:

```
1. curl localhost:8080/tasks
        ↓
   [Port Forwarding: Docker mapuje 8080→80]
        ↓

2. nginx:80 w kontenerze odbiera request
        ↓
   [Reverse Proxy: nginx robi proxy_pass]
        ↓

3. app:8000 w sieci bridge
        ↓
   [Bridge Network: nginx ↔ app komunikacja]
        ↓

4. FastAPI przetwarza request
        ↓
   Response wraca tą samą drogą
        ↓

5. curl dostaje {"tasks": {...}}
```

### Dlaczego FastAPI NIE jest dostępny bezpośrednio?

**docker-compose.yml:**
```yaml
app:
  expose:
    - "8000"  # ← EXPOSE (nie PORTS!)
  # NIE MA: ports: "8000:8000"
```

**Różnica:**
- `expose: 8000` = dostępny TYLKO w sieci Docker (app_network)
- `ports: 8000:8000` = dostępny z hosta (`curl localhost:8000`)

**Security by design:**
- ✅ `curl localhost:8080` → nginx (publiczny)
- ❌ `curl localhost:8000` → nie działa (app prywatny)
- ✅ nginx → `app:8000` → działa (wewnątrz sieci)

**To jest produkcyjny pattern!** App jest chroniony przez nginx.

---

## Quick Start

### 1. Uruchom

```bash
cd 20/
docker compose up --build
```

**Output:**
```
fastapi_nginx  | Nginx started
fastapi_app    | Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
fastapi_app    | Started 2 worker processes
```

### 2. Test

```bash
# Health check
curl http://localhost:8080/health
# → {"status": "healthy"}

# CRUD API
curl http://localhost:8080/tasks
curl -X POST http://localhost:8080/tasks -H "Content-Type: application/json" -d '{"name": "Task 3"}'
```

### 3. Logi

```bash
# Follow logs (wszystkie serwisy)
docker compose logs -f

# Tylko nginx
docker compose logs -f nginx

# JSON logs na hoście
tail -f logs/nginx/access.log
```

---

## Koncepcja 1: Nginx jako Reverse Proxy

### Czym jest reverse proxy?

```
┌────────┐       ┌───────┐       ┌─────────┐
│ Client │ ───→ │ Nginx │ ───→ │ FastAPI │
└────────┘       └───────┘       └─────────┘
                     ↑
              Jeden entry point
              Load balancing
              SSL termination
              Rate limiting
              Caching
```

**Klient nie widzi FastAPI bezpośrednio!** Tylko nginx.

### nginx.conf - Kluczowe sekcje

**Upstream (backend servers):**
```nginx
upstream fastapi_backend {
    server app:8000;  # Docker service name
    keepalive 32;     # Reuse connections
}
```

**Proxy do FastAPI:**
```nginx
location / {
    proxy_pass http://fastapi_backend;
    proxy_set_header Host $host;
    proxy_set_header X-Real-IP $remote_addr;
    proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
}
```

**Rate limiting:**
```nginx
limit_req_zone $binary_remote_addr zone=api_limit:10m rate=10r/s;

location / {
    limit_req zone=api_limit burst=20 nodelay;
}
```
- `rate=10r/s` - max 10 requestów/s per IP
- `burst=20` - pozwól na spike (20 req naraz)

### Co daje nginx?

| Feature | Opis | Dlaczego? |
|---------|------|----------|
| **Load balancing** | Rozdziela ruch między wiele instancji | Skalowanie horyzontalne |
| **SSL termination** | HTTPS → HTTP | Jeden certyfikat, FastAPI nie musi obsługiwać SSL |
| **Static files** | Serwuje /static/ | Szybciej niż FastAPI |
| **Rate limiting** | 10 req/s per IP | Ochrona przed abuse |
| **Compression** | Gzip responses | Mniej bandwidth |
| **Security headers** | X-Frame-Options, etc. | Zabezpieczenia XSS/clickjacking |

### Dlaczego nie bezpośrednio FastAPI?

**Można**, ale:
- ❌ Brak load balancing (tylko 1 instancja)
- ❌ Brak rate limiting (łatwy DoS)
- ❌ Brak SSL termination (FastAPI musi obsługiwać certyfikaty)
- ❌ Wolniejsze static files
- ❌ Brak centralizacji (każda instancja ma swój config)

**W produkcji: ZAWSZE używaj reverse proxy** (nginx, Traefik, Caddy)

---

## Koncepcja 2: Uvicorn Workers

### Dockerfile CMD

```dockerfile
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
```

### Co to znaczy `--workers 2`?

```
┌─────────────────────────┐
│  Uvicorn Master Process │
└────────┬────────────────┘
         │
    ┌────┴────┐
    │         │
┌───▼──┐  ┌──▼───┐
│Worker│  │Worker│  ← 2 worker processes
│  1   │  │  2   │
└──────┘  └──────┘
```

- Każdy worker = **osobny proces Python**
- Każdy worker obsługuje **1 request naraz**
- 2 workers = **2 równoległe requesty**

### Przykład

```python
import time

@app.get("/slow")
def slow_endpoint():
    time.sleep(5)  # Symuluj wolny request
    return {"done": True}
```

**Scenariusz 1: 1 worker**
```bash
# Terminal 1
curl http://localhost:8080/slow  # Zaczyna o 10:00:00, kończy 10:00:05

# Terminal 2 (podczas gdy Terminal 1 czeka)
curl http://localhost:8080/slow  # CZEKA aż Terminal 1 się skończy!
                            # Zaczyna o 10:00:05, kończy 10:00:10
```
**Total time: 10 sekund**

**Scenariusz 2: 2 workers**
```bash
# Terminal 1
curl http://localhost:8080/slow  # Worker 1, zaczyna 10:00:00, kończy 10:00:05

# Terminal 2 (podczas gdy Terminal 1 czeka)
curl http://localhost:8080/slow  # Worker 2, zaczyna 10:00:00, kończy 10:00:05
                            # RÓWNOLEGLE!
```
**Total time: 5 sekund** (równolegle)

### Ile workers?

**Reguły:**
- **CPU-bound** (ciężkie obliczenia): `CPU cores`
- **I/O-bound** (database, API calls): `CPU cores * 2 + 1`

**Przykłady:**
- 2 CPU cores, I/O-bound: `2 * 2 + 1 = 5 workers`
- 4 CPU cores, CPU-bound: `4 workers`

**W tym przykładzie:**
- 2 workers (prostota dla demo)
- W produkcji: 4-8 workers

### Gunicorn vs Uvicorn

**Uvicorn:**
```bash
uvicorn main:app --workers 4
```
- Proste
- Brak graceful restart

**Gunicorn + Uvicorn workers (BETTER):**
```bash
gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker
```
- Graceful restart (zero-downtime)
- Process management
- Pre-fork model

**Dla produkcji: użyj Gunicorn!**

---

## Workers - Weryfikacja i Strategie Skalowania

### Jak sprawdzić że mam 2 workery?

**Metoda 1: Wejdź do kontenera i sprawdź procesy**

```bash
# Wejdź do kontenera
docker-compose exec app sh

# Lista procesów Python
ps aux | grep python
```

**Output:**
```
PID   USER     COMMAND
1     appuser  /usr/local/bin/python -m uvicorn main:app --workers 2  ← Master
8     appuser  /usr/local/bin/python -m uvicorn main:app              ← Worker 1
9     appuser  /usr/local/bin/python -m uvicorn main:app              ← Worker 2
```

**Co to znaczy?**
- **PID 1** (Master) - główny proces, zarządza workerami
- **PID 8, 9** (Workers) - faktyczne procesy przetwarzające requesty
- **2 workers = 2 równoległe requesty naraz**

---

**Metoda 2: Logi podczas startu**

```bash
docker-compose up
```

**Output:**
```
fastapi_app | INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
fastapi_app | INFO:     Started parent process [1]
fastapi_app | INFO:     Started server process [8]  ← Worker 1
fastapi_app | INFO:     Started server process [9]  ← Worker 2
fastapi_app | INFO:     Waiting for application startup.
fastapi_app | INFO:     Application startup complete.
```

---

**Metoda 3: Test równoległych requestów**

```bash
# Terminal 1
time curl http://localhost:8080/slow

# Terminal 2 (podczas gdy Terminal 1 czeka)
time curl http://localhost:8080/slow
```

**Jeśli 2 workery:**
- Oba requesty kończą się ~5s (równolegle) ✅

**Jeśli 1 worker:**
- Pierwszy: ~5s
- Drugi: ~10s (czekał na pierwszy) ❌

---

### 3 różne podejścia do workers/skalowania

#### Podejście 1: `uvicorn --workers N` (obecny setup)

**Dockerfile:**
```dockerfile
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--workers", "2"]
```

**Architektura:**
```
┌─────────────────────────┐
│  1 Docker Container     │
│  ┌───────────────────┐  │
│  │ Uvicorn Master    │  │
│  └────┬──────────────┘  │
│       │                 │
│  ┌────┴────┐            │
│  │         │            │
│  ▼         ▼            │
│ Worker1  Worker2        │ ← 2 procesy Python
└─────────────────────────┘
```

**Charakterystyka:**
- ✅ Proste (jeden kontener)
- ✅ Działa od razu
- ❌ Brak graceful restart (wszystkie workery restart naraz)
- ❌ Jeśli kontener umrze, wszystkie workery umierają

**Kiedy używać:**
- Development
- Małe aplikacje
- Proste deployment (VPS, single server)

---

#### Podejście 2: `gunicorn` z `UvicornWorker` (production recommended)

**Dockerfile:**
```dockerfile
CMD ["gunicorn", "main:app", \
     "-w", "4", \
     "-k", "uvicorn.workers.UvicornWorker", \
     "--bind", "0.0.0.0:8000"]
```

**requirements.txt:**
```
gunicorn==23.0.0
uvicorn[standard]==0.32.1
```

**Architektura:**
```
┌─────────────────────────┐
│  1 Docker Container     │
│  ┌───────────────────┐  │
│  │ Gunicorn Master   │  │ ← Process manager
│  └────┬──────────────┘  │
│       │                 │
│  ┌────┴─────┬─────┬────┐
│  │          │     │    │
│  ▼          ▼     ▼    ▼
│ Worker1  Worker2  W3  W4│ ← 4 procesy Uvicorn
└─────────────────────────┘
```

**Charakterystyka:**
- ✅ **Graceful restart** - workery restartują się po kolei (zero-downtime)
- ✅ **Pre-fork model** - workery tworzone przed pierwszym requestem
- ✅ **Process monitoring** - gunicorn restartuje crashed workers
- ✅ **Configurable timeouts** - worker timeout, graceful timeout
- ❌ Nieco bardziej skomplikowane

**Graceful restart - jak to działa:**
```bash
# Wyślij SIGHUP do gunicorn
docker-compose exec app kill -HUP 1

# Gunicorn:
# 1. Tworzy nowe workery (W5, W6, W7, W8)
# 2. Nowe requesty idą do W5-W8
# 3. Stare workery (W1-W4) kończą aktualne requesty
# 4. Stare workery zamykają się
# ✅ Zero dropped requests!
```

**Kiedy używać:**
- **Production (RECOMMENDED!)**
- Aplikacje gdzie downtime jest niedopuszczalny
- High-traffic aplikacje

**Migracja z uvicorn na gunicorn:**
```diff
# Dockerfile
-CMD ["uvicorn", "main:app", "--workers", "2"]
+CMD ["gunicorn", "main:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", "--bind", "0.0.0.0:8000"]

# requirements.txt
+gunicorn==23.0.0
```

---

#### Podejście 3: Docker Compose `replicas` (high availability)

**docker-compose.yml:**
```yaml
services:
  app:
    image: fastapi_app:latest
    deploy:
      replicas: 3  # ← 3 osobne kontenery!
    # Każdy kontener: 1 worker (lub --workers 2)
```

**Architektura:**
```
               ┌────────────┐
               │   Nginx    │
               └──────┬─────┘
                      │
       ┌──────────────┼──────────────┐
       │              │              │
       ▼              ▼              ▼
┌────────────┐ ┌────────────┐ ┌────────────┐
│Container 1 │ │Container 2 │ │Container 3 │
│  Worker 1  │ │  Worker 1  │ │  Worker 1  │
└────────────┘ └────────────┘ └────────────┘
     Host 1         Host 2         Host 3  ← Mogą być na różnych serwerach!
```

**Charakterystyka:**
- ✅ **High Availability** - jeśli kontener 1 umrze, kontenery 2-3 działają
- ✅ **Load balancing** - nginx rozdziela ruch między repliki
- ✅ **Rolling updates** - update po kolei (zero-downtime)
- ✅ **Scale horizontal** - można dodać więcej kontenerów
- ❌ Bardziej skomplikowane (Kubernetes, Docker Swarm)
- ❌ Wymaga orchestratora

**Kiedy używać:**
- **Production z high availability**
- Aplikacje gdzie jeden server to za mało
- Microservices architecture
- Kubernetes/Cloud deployment

**Przykład - Kubernetes:**
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fastapi-app
spec:
  replicas: 3
  template:
    spec:
      containers:
      - name: app
        image: fastapi:latest
        command: ["uvicorn", "main:app", "--workers", "2"]
        # 3 kontenery × 2 workery = 6 równoległych requestów
```

---

### Porównanie strategii

| Strategia | Workers | Kontenery | Graceful restart | HA | Complexity | Use case |
|-----------|---------|-----------|------------------|-----|------------|----------|
| **uvicorn --workers 2** | 2 | 1 | ❌ | ❌ | ⭐ | Development, VPS |
| **gunicorn -w 4** | 4 | 1 | ✅ | ❌ | ⭐⭐ | **Production (single server)** |
| **replicas: 3** | 1-2/kontener | 3+ | ✅ | ✅ | ⭐⭐⭐ | **Production (HA, cloud)** |

---

### Co wybrać?

**Development (localhost, testing):**
```dockerfile
CMD ["uvicorn", "main:app", "--reload", "--workers", "1"]
```
- Proste, hot reload
- 1 worker (łatwiej debugować)

**Production - pojedynczy serwer (VPS, EC2):**
```dockerfile
CMD ["gunicorn", "main:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker"]
```
- Graceful restart
- 4 workery (wykorzystaj CPU)
- Zero-downtime deployment

**Production - cluster (Kubernetes, Docker Swarm):**
```yaml
# docker-compose.yml / Kubernetes
deploy:
  replicas: 3
```
```dockerfile
# Każdy kontener
CMD ["gunicorn", "main:app", "-w", "2", "-k", "uvicorn.workers.UvicornWorker"]
```
- 3 kontenery × 2 workery = 6 równoległych requestów
- High availability
- Automatic recovery

---

### Ile workers ustawić?

**Reguły:**
- **CPU-bound** (ciężkie obliczenia): `CPU cores`
- **I/O-bound** (database, API calls): `CPU cores × 2 + 1`

**Przykłady:**

**VPS z 2 CPU cores (FastAPI z PostgreSQL - I/O-bound):**
```
2 × 2 + 1 = 5 workers
```
```dockerfile
CMD ["gunicorn", "main:app", "-w", "5", "-k", "uvicorn.workers.UvicornWorker"]
```

**Kubernetes z 3 replicas, każda 1 CPU core:**
```
3 replicas × (1 × 2 + 1) workers = 9 total workers
```
```yaml
replicas: 3
```
```dockerfile
CMD ["gunicorn", "main:app", "-w", "3", "-k", "uvicorn.workers.UvicornWorker"]
```

---

### Weryfikacja setup

```bash
# 1. Sprawdź procesy
docker-compose exec app ps aux

# 2. Sprawdź równoległość
# Terminal 1
time curl http://localhost:8080/slow

# Terminal 2 (podczas)
time curl http://localhost:8080/slow

# Jeśli oba ~5s → workery działają równolegle ✅
# Jeśli drugi ~10s → tylko 1 worker ❌

# 3. Load test
# Zainstaluj: pip install locust
locust -f locustfile.py --host http://localhost:8080
```

**locustfile.py:**
```python
from locust import HttpUser, task

class FastAPIUser(HttpUser):
    @task
    def get_tasks(self):
        self.client.get("/tasks")
```

**Output (50 users, 2 workers):**
```
Requests/s: ~200
95th percentile: 250ms
```

**Output (50 users, 4 workers - gunicorn):**
```
Requests/s: ~400  ← 2x więcej!
95th percentile: 120ms
```

---

## Koncepcja 3: Multi-stage Docker Build

### Dockerfile

```dockerfile
# ──────────────────────────────────────────
# Stage 1: Builder
# ──────────────────────────────────────────
FROM python:3.11-slim AS builder

WORKDIR /build

# Zainstaluj build dependencies
RUN apt-get update && apt-get install -y build-essential

# Zainstaluj Python packages
COPY requirements.txt .
RUN pip install --user -r requirements.txt

# ──────────────────────────────────────────
# Stage 2: Runtime
# ──────────────────────────────────────────
FROM python:3.11-slim

WORKDIR /app

# Skopiuj TYLKO zainstalowane packages (nie build tools!)
COPY --from=builder /root/.local /home/appuser/.local

# Skopiuj kod
COPY main.py .

# Non-root user (security)
RUN useradd -m appuser
USER appuser

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--workers", "2"]
```

### Dlaczego multi-stage?

**Single-stage (BEZ multi-stage):**
```dockerfile
FROM python:3.11
RUN apt-get install build-essential gcc  # Build tools
RUN pip install fastapi uvicorn
COPY main.py .
CMD ["uvicorn", "main:app"]
```
**Rozmiar:** ~500MB ❌
- Zawiera: Python + build tools (gcc, make) + libraries + kod
- Build tools niepotrzebne w runtime!

**Multi-stage:**
```dockerfile
# Stage 1: Install (with build tools)
FROM python:3.11-slim AS builder
RUN pip install --user fastapi uvicorn

# Stage 2: Copy only installed packages
FROM python:3.11-slim
COPY --from=builder /root/.local /home/appuser/.local
COPY main.py .
CMD ["uvicorn", "main:app"]
```
**Rozmiar:** ~150MB ✅
- Zawiera: Python + installed packages + kod
- NIE zawiera: build tools, cache, intermediate files

**Korzyści:**
- ✅ **3x mniejszy obraz** (~150MB vs ~500MB)
- ✅ **Szybszy deploy** (mniej do uploadowania)
- ✅ **Bezpieczniejszy** (brak gcc = trudniej exploit)
- ✅ **Czystszy** (tylko runtime dependencies)

### Non-root user (security)

**Dlaczego nie root?**
```dockerfile
# ❌ ZŁE - app działa jako root
CMD ["uvicorn", "main:app"]
```
Jeśli attacker wejdzie do kontenera (exploit), ma **root access**!

```dockerfile
# ✅ DOBRE - app działa jako appuser (uid 1000)
RUN useradd -m appuser
USER appuser
CMD ["uvicorn", "main:app"]
```
Jeśli attacker wejdzie, ma **limited permissions** (nie może instalować packages, zmieniać system files).

---

## Koncepcja 4: Health Checks

### Dockerfile HEALTHCHECK

```dockerfile
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
```

**Co to robi?**
- Co 30s Docker wywołuje `/health` endpoint
- Jeśli response w < 10s → **healthy**
- Jeśli 3 próby fail → **unhealthy**

**Po co?**
- Docker wie czy kontener działa
- Load balancer może wyłączyć unhealthy instance
- Orchestrator (Kubernetes) może zrestartować unhealthy pod

### FastAPI Health Endpoint

```python
@app.get("/health")
def health_check():
    """Health check endpoint"""
    return {"status": "healthy"}
```

**Zaawansowany health check:**
```python
from sqlalchemy import text

@app.get("/health")
def health_check(db: Session = Depends(get_db)):
    # Sprawdź połączenie do bazy
    try:
        db.execute(text("SELECT 1"))
        return {"status": "healthy", "database": "ok"}
    except Exception as e:
        raise HTTPException(500, f"Database unhealthy: {e}")
```

### docker-compose depends_on

```yaml
services:
  nginx:
    depends_on:
      app:
        condition: service_healthy  # Czekaj aż app będzie healthy!
```

**Co to daje?**
- Nginx NIE startuje dopóki FastAPI nie jest ready
- Unika 502 Bad Gateway podczas startu

**Bez health check:**
```
10:00:00 - nginx starts (FastAPI jeszcze nie działa)
10:00:01 - klient: curl http://localhost:8080/tasks → 502 Bad Gateway ❌
10:00:05 - FastAPI ready
10:00:06 - klient: curl http://localhost:8080/tasks → 200 OK ✅
```

**Z health check:**
```
10:00:00 - FastAPI starting...
10:00:05 - FastAPI healthy ✅
10:00:05 - nginx starts (FastAPI ready!)
10:00:06 - klient: curl http://localhost:8080/tasks → 200 OK ✅
```

---

## Koncepcja 5: Structured Logging (JSON)

### nginx.conf log format

```nginx
log_format json_combined escape=json
'{'\ n    '"time_local":"$time_local",'\ n    '"remote_addr":"$remote_addr",'\ n    '"request":"$request",'\ n    '"status": "$status",'\ n    '"body_bytes_sent":"$body_bytes_sent",'\ n    '"request_time":"$request_time"'
'}';

access_log /var/log/nginx/access.log json_combined;
```

### Przykładowy log (JSON)

**Standard nginx log (text):**
```
172.18.0.1 - - [16/Jun/2026:10:30:15 +0000] "GET /tasks HTTP/1.1" 200 145 "-" "curl/7.68.0"
```
❌ Trudno parsować (regex, split)

**JSON log:**
```json
{
  "time_local": "16/Jun/2026:10:30:15 +0000",
  "remote_addr": "172.18.0.1",
  "request": "GET /tasks HTTP/1.1",
  "status": "200",
  "body_bytes_sent": "145",
  "request_time": "0.012"
}
```
✅ Łatwo parsować (jq, Python, Elasticsearch)

### Parsowanie logów

**jq - wszystkie 404 errory:**
```bash
cat logs/nginx/access.log | jq 'select(.status == "404")'
```

**jq - średni request time:**
```bash
cat logs/nginx/access.log | jq '.request_time | tonumber' | \
  awk '{sum+=$1; count++} END {print sum/count}'
```

**Python - agregacja:**
```python
import json
from collections import Counter

status_codes = Counter()

with open('logs/nginx/access.log') as f:
    for line in f:
        log = json.loads(line)
        status_codes[log['status']] += 1

print(status_codes)
# Counter({'200': 1450, '404': 23, '500': 5})
```

### Dlaczego JSON?

| Format | Parsing | Agregacja | Tools |
|--------|---------|-----------|-------|
| **Text** | Regex ❌ | Awk/sed ❌ | grep, tail |
| **JSON** | Native ✅ | Python/jq ✅ | Elasticsearch, Loki, Grafana |

**W produkcji:**
- Logi JSON → Elasticsearch/Loki
- Grafana dashboards (request rate, errors, latency)
- Alerting (Slack/PagerDuty jeśli error rate > 5%)

---

## Debugging

### Problem 1: App nie startuje

**Symptom:**
```bash
docker-compose up
# fastapi_app exited with code 1
```

**Debug:**
```bash
# Zobacz logi
docker-compose logs app

# Wejdź do kontenera
docker-compose run --rm app bash

# Test uvicorn manualnie
uvicorn main:app --host 0.0.0.0 --port 8000
```

**Najczęstsze przyczyny:**
- Import error (brak dependency w requirements.txt)
- Syntax error w main.py
- Port już zajęty

### Problem 2: Nginx zwraca 502 Bad Gateway

**Symptom:**
```bash
curl http://localhost:8080/tasks
# <html>502 Bad Gateway</html>
```

**Przyczyny:**
1. **FastAPI nie działa**
   ```bash
   docker-compose ps
   # app - Exit 1 (unhealthy)
   ```

2. **Błędny upstream w nginx.conf**
   ```nginx
   upstream fastapi_backend {
       server app:8000;  # ← Poprawny Docker service name!
   }
   ```

3. **Health check failuje**
   ```bash
   docker-compose exec app curl http://localhost:8000/health
   # curl: (7) Failed to connect
   ```

**Fix:**
```bash
# Restart app
docker-compose restart app

# Sprawdź logi
docker-compose logs app nginx
```

### Problem 3: Requesty wolne

**Symptom:**
```bash
curl http://localhost:8080/tasks
# Czeka 5 sekund...
```

**Debug:**
```bash
# Sprawdź request_time w logach
tail -f logs/nginx/access.log | jq '.request_time'
# "5.234"  ← 5 sekund!

# Sprawdź zasoby
docker stats
# fastapi_app - 98% CPU, 450MB RAM ← Problem!
```

**Możliwe przyczyny:**
- Wolne query do bazy (brak indexów)
- N+1 queries
- Zbyt mało workers (zwiększ w Dockerfile)
- Memory leak

**Fix:**
```bash
# Zwiększ workers
# Dockerfile: --workers 4

# Dodaj indexes w bazie
# CREATE INDEX idx_tasks_user_id ON tasks(user_id);

# Profile code (cProfile, py-spy)
```

### Problem 4: Brak logów

**Symptom:**
```bash
ls logs/nginx/
# empty ❌
```

**Przyczyna:**
Volume nie jest zmontowany lub permissions.

**Fix:**
```bash
# Sprawdź volumes
docker-compose config | grep volumes

# Sprawdź permissions
ls -la logs/

# Fix permissions
chmod -R 755 logs/
```

---

## Podsumowanie

### Co nauczyliśmy się?

| Koncepcja | Dlaczego? | Production benefit |
|-----------|-----------|--------------------|
| **Nginx reverse proxy** | Load balancing, SSL, rate limiting | Skalowanie, bezpieczeństwo |
| **Uvicorn workers** | Równoległe przetwarzanie | Więcej requestów/s |
| **Multi-stage build** | Mniejszy obraz | Szybszy deploy |
| **Health checks** | Automatic recovery | Zero-downtime |
| **JSON logs** | Łatwe parsowanie | Monitoring, alerting |

### Development vs Production

| Aspekt | Development | Production |
|--------|-------------|------------|
| **Proxy** | Brak (direct uvicorn) | **Nginx** reverse proxy |
| **Workers** | 1 (--reload) | **4-8** (no reload) |
| **Logs** | stdout | **JSON files** + aggregation |
| **SSL** | HTTP | **HTTPS** (Let's Encrypt) |
| **Secrets** | Hardcoded | **.env** / Vault |
| **Database** | SQLite | **PostgreSQL** + replicas |
| **Monitoring** | Brak | **Prometheus/Grafana** |
| **Deploy** | Manual | **CI/CD** (GitHub Actions) |

### Kluczowe takeaways

**1. NIGDY nie expose FastAPI bezpośrednio na internet**
- Zawsze używaj reverse proxy (nginx, Traefik, Caddy)

**2. Multiple workers są KLUCZOWE**
- 1 worker = 1 request naraz (bottleneck)
- 4+ workers = lepsze wykorzystanie CPU

**3. Health checks są obowiązkowe**
- Docker/Kubernetes wie czy app działa
- Automatic restarts

**4. JSON logs > text logs**
- Łatwe parsowanie
- Integracja z Elasticsearch/Loki
- Dashboards w Grafana

**5. Multi-stage build = smaller images**
- 3x mniejszy obraz
- Szybszy deploy
- Bezpieczniejszy

---

## Next Steps (Advanced)

Gdy chcesz rozwijać dalej:

### 1. Gunicorn zamiast Uvicorn

```dockerfile
CMD ["gunicorn", "main:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", 
     "--bind", "0.0.0.0:8000"]
```

**Korzyści:**
- Graceful restart (zero-downtime)
- Pre-fork model
- Better process management

### 2. SSL/TLS (HTTPS)

```nginx
server {
    listen 443 ssl;
    ssl_certificate /etc/letsencrypt/live/yourdomain.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/yourdomain.com/privkey.pem;
}
```

**Certbot (Let's Encrypt):**
```bash
certbot --nginx -d yourdomain.com
```

### 3. PostgreSQL + Connection Pooling

```python
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

engine = create_engine(
    DATABASE_URL,
    poolclass=QueuePool,
    pool_size=10,        # Max 10 connections
    max_overflow=20,     # Extra 20 jeśli pool pełny
    pool_pre_ping=True   # Check connection przed użyciem
)
```

### 4. Redis Caching

```python
import redis
from functools import wraps

redis_client = redis.Redis(host='redis', port=6379)

def cache(ttl=60):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            key = f"{func.__name__}:{args}:{kwargs}"
            cached = redis_client.get(key)
            if cached:
                return json.loads(cached)
            result = func(*args, **kwargs)
            redis_client.setex(key, ttl, json.dumps(result))
            return result
        return wrapper
    return decorator

@app.get("/tasks")
@cache(ttl=300)  # Cache na 5 minut
def get_tasks():
    # Expensive database query
    return tasks
```

### 5. Prometheus + Grafana Monitoring

```python
from prometheus_client import Counter, Histogram

request_count = Counter('http_requests_total', 'Total HTTP requests')
request_duration = Histogram('http_request_duration_seconds', 'HTTP request duration')

@app.middleware("http")
async def prometheus_middleware(request, call_next):
    with request_duration.time():
        response = await call_next(request)
    request_count.inc()
    return response
```

### 6. CI/CD (GitHub Actions)

```yaml
# .github/workflows/deploy.yml
name: Deploy
on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v2
      - name: Build Docker image
        run: docker build -t myapp .
      - name: Push to registry
        run: docker push myapp:latest
      - name: Deploy to production
        run: ssh server 'docker-compose up -d'
```

### 7. Kubernetes (Orchestration)

```yaml
# deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: fastapi-app
spec:
  replicas: 3  # 3 instancje
  template:
    spec:
      containers:
      - name: app
        image: myapp:latest
        ports:
        - containerPort: 8000
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
```

---

## Materiały

- [Nginx Reverse Proxy Guide](https://docs.nginx.com/nginx/admin-guide/web-server/reverse-proxy/)
- [Uvicorn Deployment](https://www.uvicorn.org/deployment/)
- [FastAPI Deployment](https://fastapi.tiangolo.com/deployment/)
- [Docker Multi-stage Builds](https://docs.docker.com/build/building/multi-stage/)
- [Gunicorn Settings](https://docs.gunicorn.org/en/stable/settings.html)

---